In [15]:
# 1. Import Libraries
import pandas as pd

In [16]:
# 2. Load Datasets
ratings = pd.read_csv(
    "data/tmdb_movie_ratings.csv"
)

# Limit users
ratings = ratings[
    ratings["userId"] <= 600
].copy()
print("Ratings Shape:", ratings.shape)

movies = pd.read_csv(
    "data/tmdb_movie_dataset.csv",
    usecols=[
        "ratingId",
        "tmdbId",
        "title",
        "vote_average",
        "vote_count"
    ]
)

print("Ratings Dataset:")
display(ratings.head())

print("\nMovie Dataset:")
display(movies.head())

Ratings Shape: (55511, 4)
Ratings Dataset:


,userId,ratingId,rating,timestamp
21362,312,1,4.0,2011-08-22 17:35:35
21363,304,1,4.0,1999-10-04 11:25:43
21364,302,1,5.0,2016-08-10 19:19:36
21365,301,1,3.0,2017-01-17 18:11:16
21366,298,1,5.0,1997-02-10 15:23:28



Movie Dataset:


,tmdbId,title,vote_average,vote_count,ratingId
0,5,Four Rooms,6.5,530,18
1,11,Star Wars,8.1,6624,260
2,12,Finding Nemo,7.6,6122,6377
3,13,Forrest Gump,8.2,7927,356
4,14,American Beauty,7.9,3313,2858


In [17]:
# 3. Check Columns
print("Ratings Columns:")
print(ratings.columns.tolist())

print("\nMovie Dataset Columns:")
print(movies.columns.tolist())

Ratings Columns:
['userId', 'ratingId', 'rating', 'timestamp']

Movie Dataset Columns:
['tmdbId', 'title', 'vote_average', 'vote_count', 'ratingId']


In [18]:
# 4. Merge Ratings with Dataset
merged_data = pd.merge(
    ratings,
    movies,
    on="ratingId",
    how="inner"
)

print("Merged Dataset:")
display(
    merged_data[
        [
            "userId",
            "ratingId",
            "rating",
            "tmdbId",
            "title",
            "vote_average",
            "vote_count"
        ]
    ].head()
)

print("Shape:", merged_data.shape)

Merged Dataset:


,userId,ratingId,rating,tmdbId,title,vote_average,vote_count
0,312,1,4.0,862,Toy Story,7.7,5269
1,304,1,4.0,862,Toy Story,7.7,5269
2,302,1,5.0,862,Toy Story,7.7,5269
3,301,1,3.0,862,Toy Story,7.7,5269
4,298,1,5.0,862,Toy Story,7.7,5269


Shape: (55511, 8)


In [19]:
# 5. Create Collaborative Filtering Dataset
cf_data = merged_data[
    [
        "userId",
        "ratingId",
        "tmdbId",
        "title",
        "rating"
    ]
].copy()

print("Collaborative Filtering Dataset:")
display(cf_data.head())

print("Shape:", cf_data.shape)


Collaborative Filtering Dataset:


,userId,ratingId,tmdbId,title,rating
0,312,1,862,Toy Story,4.0
1,304,1,862,Toy Story,4.0
2,302,1,862,Toy Story,5.0
3,301,1,862,Toy Story,3.0
4,298,1,862,Toy Story,5.0


Shape: (55511, 5)


In [20]:
# 6. Movie Statistics
movie_stats = pd.DataFrame(
    cf_data.groupby("title")["rating"].mean()
)

movie_stats["number of ratings"] = (
    cf_data.groupby("title")["rating"].count()
)

display(movie_stats.head())

,rating,number of ratings
title,,
(500) Days of Summer,3.900000,40
10 Cloverfield Lane,4.000000,12
10 Things I Hate About You,3.846939,49
102 Dalmatians,1.750000,6
11:14,3.166667,3


In [21]:
# 7. Movies with Most User Ratings
top_rated_movies = (
    movie_stats
    .sort_values(
        "number of ratings",
        ascending=False
    )
    .head(10)
)

display(top_rated_movies)

,rating,number of ratings
title,,
Forrest Gump,4.074074,297
Pulp Fiction,4.137324,284
The Shawshank Redemption,4.474820,278
The Silence of the Lambs,4.207692,260
The Matrix,4.179134,254
Jurassic Park,3.702083,240
Star Wars,4.188841,233
Schindler's List,4.375000,228
Braveheart,4.102439,205


In [22]:
# 8. Create User-Movie Matrix
movie_matrix = cf_data.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

display(movie_matrix.head())

title,(500) Days of Summer,10 Cloverfield Lane,10 Things I Hate About You,102 Dalmatians,11:14,12 Angry Men,12 Rounds,12 Years a Slave,127 Hours,13 Going on 30,...,Zombieland,Zookeeper,Zoolander,Zoolander 2,[REC],[REC]²,eXistenZ,xXx,xXx: State of the Union,Æon Flux
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,NaN,3.5,NaN,NaN,NaN,NaN,3.5,NaN,4.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# 9. Enter Favorite Movie
favorite_movie = input(
    "Please enter your favorite movie: "
)

print("Selected Movie:", favorite_movie)

Selected Movie: Toy Story


In [24]:
# 10. Check Favorite Movie
if favorite_movie not in movie_matrix.columns:

    print("Movie not found.")

    matches = [
        movie
        for movie in movie_matrix.columns
        if favorite_movie.lower() in movie.lower()
    ]

    if matches:
        print("\nPossible matches:")

        for movie in matches[:10]:
            print("-", movie)

else:
    print("Movie found:", favorite_movie)

Movie found: Toy Story


In [25]:
# 11. Calculate Movie Correlation
if favorite_movie in movie_matrix.columns:

    similar = movie_matrix.corrwith(
        movie_matrix[favorite_movie]
    )

    corr = pd.DataFrame(
        similar,
        columns=["Correlation"]
    )

    corr.dropna(
        inplace=True
    )

    display(corr.head())

c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\ngjiu\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


,Correlation
title,
(500) Days of Summer,0.295914
10 Cloverfield Lane,1.000000
10 Things I Hate About You,0.374959
102 Dalmatians,-0.944911
12 Angry Men,0.245865


In [26]:
# 12. Add Number of User Ratings
if favorite_movie in movie_matrix.columns:

    corr = corr.join(
        movie_stats[
            ["number of ratings"]
        ]
    )

    display(corr.head())

,Correlation,number of ratings
title,,
(500) Days of Summer,0.295914,40
10 Cloverfield Lane,1.000000,12
10 Things I Hate About You,0.374959,49
102 Dalmatians,-0.944911,6
12 Angry Men,0.245865,54


In [27]:
# 13. Filter Movies
if favorite_movie in movie_matrix.columns:

    recommendations = corr[
        corr["number of ratings"] >= 100
    ].copy()

    recommendations = (
        recommendations
        .drop(favorite_movie)
        .sort_values(
            "Correlation",
            ascending=False
        )
    )

    display(
        recommendations.head(10)
    )

,Correlation,number of ratings
title,,
"Monsters, Inc.",0.623669,122
Finding Nemo,0.619362,116
Babe,0.485719,106
Four Weddings and a Funeral,0.452463,106
Home Alone,0.433749,101
One Flew Over the Cuckoo's Nest,0.421353,119
Amélie,0.420297,112
Back to the Future,0.413103,174
Shrek,0.408669,156


In [28]:
# 14. Top 10 Recommendations
if favorite_movie in movie_matrix.columns:

    top_recommendations = (
        recommendations
        .head(10)
        .reset_index()
    )

    top_recommendations = (
        top_recommendations
        .rename(
            columns={
                "title": "Movie Title",
                "Correlation": "Correlation Score",
                "number of ratings": "Number of User Ratings"
            }
        )
    )

    display(top_recommendations)

,Movie Title,Correlation Score,Number of User Ratings
0,"Monsters, Inc.",0.623669,122
1,Finding Nemo,0.619362,116
2,Babe,0.485719,106
3,Four Weddings and a Funeral,0.452463,106
4,Home Alone,0.433749,101
5,One Flew Over the Cuckoo's Nest,0.421353,119
6,Amélie,0.420297,112
7,Back to the Future,0.413103,174
8,Shrek,0.408669,156
9,Memento,0.407964,138
